## Week 15

### import and libraries

In [1]:
!pip install nltk
!pip install -U transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 21.9 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 4.48.3
    Uninstalling transformers-4.48.3:
      Successfully uninstalled transformers-4.48.3


In [ ]:
import importlib.util
import subprocess
import sys
def is_package_installed(package_name):
  spec = importlib.util.find_spec(package_name)
  return spec is not None
def install_package(package_name):
  subprocess.check_call([sys.executable, "-m", "pip", "install", package_name])
packages = ['keras', 'numpy', 'pandas', 'scikit-learn', 'torch', 'transformers']
for package in packages:
  if not is_package_installed(package):
    try:
      print(f"Installing {package}...")
      install_package(package)
    except subprocess.CalledProcessError as error:
     print(f"Error installing {package}: {error}")

Installing scikit-learn...


In [ ]:
import pandas as pd
import numpy as np
import numpy
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from transformers import GPT2Tokenizer
from nltk import pos_tag, ne_chunk
from nltk.tokenize import word_tokenize
from sklearn.preprocessing import LabelBinarizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import CountVectorizer
import gensim.downloader
import matplotlib.pyplot as pyplot
import seaborn
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
# import torch.nn.functional as functional
from keras.preprocessing.sequence import pad_sequences

from sklearn.metrics import matthews_corrcoef
from torch.utils.data import DataLoader, RandomSampler, SequentialSampler, TensorDataset
from transformers import AdamW, BertConfig, BertForSequenceClassification, BertModel, BertTokenizer
from tqdm import trange
from transformers import BertTokenizer, BertModel, get_scheduler, BertConfig
from sklearn.utils.class_weight import compute_class_weight
# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

nltk.download('averaged_perceptron_tagger_eng')
nltk.download('maxent_ne_chunker_tab')
nltk.download('punkt_tab')
nltk.download('words')
nltk.download('punkt_tab')
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /usr/share/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.
[nltk_data] Downloading package maxent_ne_chunker_tab to
[nltk_data]     /usr/share/nltk_data...
[nltk_data]   Unzipping chunkers/maxent_ne_chunker_tab.zip.
[nltk_data] Downloading package punkt_tab to /usr/share/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package words to /usr/share/nltk_data...
[nltk_data]   Package words is already up-to-date!
[nltk_data] Downloading package punkt_tab to /usr/share/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /usr/share/nltk_data...
[nltk_data]

True

###data cleaning###

In [ ]:
df = pd.read_csv('/kaggle/input/dataset2/TAN HAO WEN_dataset.csv')
df

,Category,Resume
0,HR,Education Details \r\n BA mumbai University\...
1,Electrical Engineering,Skills: 1) MC Office 2) AutoCAD 2016 3) Introd...
2,Advocate,Skills Legal Writing Efficient researcher Lega...
3,Electrical Engineering,Skills: 1) MC Office 2) AutoCAD 2016 3) Introd...
4,Testing,Computer Skills: â¢ Proficient in MS office (...
...,...,...
957,Blockchain,Hobbies â¢ Playing Chess â¢ Solving Rubik's ...
958,DevOps Engineer,Skills VISA B1-VISA (USA) Onsite Visits to Swe...
959,Testing,Computer Skills: â¢ Proficient in MS office (...
960,Hadoop,"Skill Set: Hadoop, Map Reduce, HDFS, Hive, Sqo..."


In [ ]:
df.shape

(962, 2)

In [ ]:
df['Category'].value_counts()

Category
Java Developer               84
Testing                      70
DevOps Engineer              55
Python Developer             48
Web Designing                45
HR                           44
Hadoop                       42
ETL Developer                40
Operations Manager           40
Blockchain                   40
Sales                        40
Mechanical Engineer          40
Data Science                 40
Arts                         36
Database                     33
Health and fitness           30
Electrical Engineering       30
PMO                          30
Business Analyst             28
DotNet Developer             28
Automation Testing           26
Network Security Engineer    25
Civil Engineer               24
SAP Developer                24
Advocate                     20
Name: count, dtype: int64

In [ ]:
df['Category'].unique()

array(['HR', 'Electrical Engineering', 'Advocate', 'Testing',
       'ETL Developer', 'Java Developer', 'Hadoop', 'Web Designing',
       'Operations Manager', 'DotNet Developer', 'Database',
       'SAP Developer', 'DevOps Engineer', 'Civil Engineer',
       'Mechanical Engineer', 'Python Developer', 'Automation Testing',
       'Data Science', 'Business Analyst', 'Blockchain', 'Arts', 'Sales',
       'PMO', 'Health and fitness', 'Network Security Engineer'],
      dtype=object)

In [ ]:
def remove_numbers_with_special_characters(text):
    text = re.sub(r'\d+\.\s*',' ', text)  # Remove numbers followed by a dot and space (e.g., "1.MC")
    text =  re.sub(r'\d+\)\s*', ' ', text) # Remove numbers followed by ' ') example (e.g., "1)MC")
    text = re.sub(r'\b\w*[a-zA-Z]+\d+[a-zA-Z]*\b', '', text)# Remove numbers followed by random characters and numbers examples (e.g., "abc123","a1b2c")
    text = re.sub(r'[^\w\s]', ' ', text)  # Remove other special characters
    text = re.sub(r'\s+', ' ', text).strip()  # Normalize spaces
    return text


In [ ]:
# Step 1: Clean the 'Resume' column and store in 'Cleaned_Resume'
df['Cleaned_Resume'] = df['Resume'].apply(remove_numbers_with_special_characters)
df

,Category,Resume,Cleaned_Resume
0,HR,Education Details \r\n BA mumbai University\...,Education Details BA mumbai University HR Skil...
1,Electrical Engineering,Skills: 1) MC Office 2) AutoCAD 2016 3) Introd...,Skills MC Office AutoCAD 2016 Introductory Kno...
2,Advocate,Skills Legal Writing Efficient researcher Lega...,Skills Legal Writing Efficient researcher Lega...
3,Electrical Engineering,Skills: 1) MC Office 2) AutoCAD 2016 3) Introd...,Skills MC Office AutoCAD 2016 Introductory Kno...
4,Testing,Computer Skills: â¢ Proficient in MS office (...,Computer Skills â Proficient in MS office Word...
...,...,...,...
957,Blockchain,Hobbies â¢ Playing Chess â¢ Solving Rubik's ...,Hobbies â Playing Chess â Solving Rubik s Cube...
958,DevOps Engineer,Skills VISA B1-VISA (USA) Onsite Visits to Swe...,Skills VISA VISA USA Onsite Visits to Sweden U...
959,Testing,Computer Skills: â¢ Proficient in MS office (...,Computer Skills â Proficient in MS office Word...
960,Hadoop,"Skill Set: Hadoop, Map Reduce, HDFS, Hive, Sqo...",Skill Set Hadoop Map Reduce HDFS Hive Sqoop ja...


In [ ]:
def remove_special_characters(text):
  text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
  text = re.sub(r'\s+', ' ', text).strip()
  return text

In [ ]:
df['Process_text'] = df['Cleaned_Resume'].apply(remove_special_characters)
df

,Category,Resume,Cleaned_Resume,Process_text
0,HR,Education Details \r\n BA mumbai University\...,Education Details BA mumbai University HR Skil...,Education Details BA mumbai University HR Skil...
1,Electrical Engineering,Skills: 1) MC Office 2) AutoCAD 2016 3) Introd...,Skills MC Office AutoCAD 2016 Introductory Kno...,Skills MC Office AutoCAD 2016 Introductory Kno...
2,Advocate,Skills Legal Writing Efficient researcher Lega...,Skills Legal Writing Efficient researcher Lega...,Skills Legal Writing Efficient researcher Lega...
3,Electrical Engineering,Skills: 1) MC Office 2) AutoCAD 2016 3) Introd...,Skills MC Office AutoCAD 2016 Introductory Kno...,Skills MC Office AutoCAD 2016 Introductory Kno...
4,Testing,Computer Skills: â¢ Proficient in MS office (...,Computer Skills â Proficient in MS office Word...,Computer Skills Proficient in MS office Word B...
...,...,...,...,...
957,Blockchain,Hobbies â¢ Playing Chess â¢ Solving Rubik's ...,Hobbies â Playing Chess â Solving Rubik s Cube...,Hobbies Playing Chess Solving Rubik s Cube Wat...
958,DevOps Engineer,Skills VISA B1-VISA (USA) Onsite Visits to Swe...,Skills VISA VISA USA Onsite Visits to Sweden U...,Skills VISA VISA USA Onsite Visits to Sweden U...
959,Testing,Computer Skills: â¢ Proficient in MS office (...,Computer Skills â Proficient in MS office Word...,Computer Skills Proficient in MS office Word B...
960,Hadoop,"Skill Set: Hadoop, Map Reduce, HDFS, Hive, Sqo...",Skill Set Hadoop Map Reduce HDFS Hive Sqoop ja...,Skill Set Hadoop Map Reduce HDFS Hive Sqoop ja...


In [ ]:
# Drop the 'Resume' and 'Cleaned_Resume' columns
df = df.drop(columns=['Resume', 'Cleaned_Resume'])

df


,Category,Process_text
0,HR,Education Details BA mumbai University HR Skil...
1,Electrical Engineering,Skills MC Office AutoCAD 2016 Introductory Kno...
2,Advocate,Skills Legal Writing Efficient researcher Lega...
3,Electrical Engineering,Skills MC Office AutoCAD 2016 Introductory Kno...
4,Testing,Computer Skills Proficient in MS office Word B...
...,...,...
957,Blockchain,Hobbies Playing Chess Solving Rubik s Cube Wat...
958,DevOps Engineer,Skills VISA VISA USA Onsite Visits to Sweden U...
959,Testing,Computer Skills Proficient in MS office Word B...
960,Hadoop,Skill Set Hadoop Map Reduce HDFS Hive Sqoop ja...


In [ ]:
def remove_stopwords(text):
    stop_words = set(stopwords.words('english'))
    words = word_tokenize(text)
    filtered_words = [word for word in words if word.lower() not in stop_words]
    return text

In [ ]:
df['Cleaned_text'] = df['Process_text'].apply(remove_stopwords)
df

,Category,Process_text,Cleaned_text
0,HR,Education Details BA mumbai University HR Skil...,Education Details BA mumbai University HR Skil...
1,Electrical Engineering,Skills MC Office AutoCAD 2016 Introductory Kno...,Skills MC Office AutoCAD 2016 Introductory Kno...
2,Advocate,Skills Legal Writing Efficient researcher Lega...,Skills Legal Writing Efficient researcher Lega...
3,Electrical Engineering,Skills MC Office AutoCAD 2016 Introductory Kno...,Skills MC Office AutoCAD 2016 Introductory Kno...
4,Testing,Computer Skills Proficient in MS office Word B...,Computer Skills Proficient in MS office Word B...
...,...,...,...
957,Blockchain,Hobbies Playing Chess Solving Rubik s Cube Wat...,Hobbies Playing Chess Solving Rubik s Cube Wat...
958,DevOps Engineer,Skills VISA VISA USA Onsite Visits to Sweden U...,Skills VISA VISA USA Onsite Visits to Sweden U...
959,Testing,Computer Skills Proficient in MS office Word B...,Computer Skills Proficient in MS office Word B...
960,Hadoop,Skill Set Hadoop Map Reduce HDFS Hive Sqoop ja...,Skill Set Hadoop Map Reduce HDFS Hive Sqoop ja...


In [ ]:
df = df.drop(columns=['Process_text'])
df

,Category,Cleaned_text
0,HR,Education Details BA mumbai University HR Skil...
1,Electrical Engineering,Skills MC Office AutoCAD 2016 Introductory Kno...
2,Advocate,Skills Legal Writing Efficient researcher Lega...
3,Electrical Engineering,Skills MC Office AutoCAD 2016 Introductory Kno...
4,Testing,Computer Skills Proficient in MS office Word B...
...,...,...
957,Blockchain,Hobbies Playing Chess Solving Rubik s Cube Wat...
958,DevOps Engineer,Skills VISA VISA USA Onsite Visits to Sweden U...
959,Testing,Computer Skills Proficient in MS office Word B...
960,Hadoop,Skill Set Hadoop Map Reduce HDFS Hive Sqoop ja...


In [ ]:
# Count duplicate rows
num_duplicates = df.duplicated().sum()
print(f"Number of duplicate rows: {num_duplicates}")

Number of duplicate rows: 778


In [ ]:
# Dropout duplicate rows
df = df.drop_duplicates()
df.shape


(184, 2)

In [ ]:
df

,Category,Cleaned_text
0,HR,Education Details BA mumbai University HR Skil...
1,Electrical Engineering,Skills MC Office AutoCAD 2016 Introductory Kno...
2,Advocate,Skills Legal Writing Efficient researcher Lega...
4,Testing,Computer Skills Proficient in MS office Word B...
5,ETL Developer,Computer skills Yes SQL knowledge yes Unix kno...
...,...,...
763,SAP Developer,Skills ETL Data Warehousing SQL PL SQL Basic C...
800,Sales,Skill Sets Multi tasking Collaborative Optimis...
850,PMO,AREA OF EXPERTISE PROFILE Around 10 plus years...
870,DevOps Engineer,Technical Skills Key Skills MS Technology Net ...


In [ ]:
# Subword tokenization
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
text = df['Cleaned_text'][0]
tokens = tokenizer.tokenize(text)

tokens

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

['Education',
 'ĠDetails',
 'ĠBA',
 'Ġm',
 'umbai',
 'ĠUniversity',
 'ĠHR',
 'ĠSkill',
 'ĠDetails',
 'ĠH',
 'r',
 'ĠOperations',
 'ĠEx',
 'pri',
 'ence',
 'ĠLess',
 'Ġthan',
 'Ġ1',
 'Ġyear',
 'Ġmonths',
 'Company',
 'ĠDetails',
 'Ġcompany',
 'ĠMumbai',
 'ĠMon',
 'or',
 'ail',
 'Ġdescription',
 'Ġstr',
 'ati',
 'osp',
 'he',
 'com',
 'y',
 'io',
 'ides']

## Week 16

###Top 10 resumes similar to your resume ###




####data preprocessing for my own CV ###

In [ ]:
file_path ='/kaggle/input/dataset2/resume.csv'
cv = pd.read_csv(file_path, header=None, encoding ='ISO-8859-1')

# Convert entire DataFrame into a single string (merge everything)
merged_text = ' '.join(cv.astype(str).values.flatten())

# Remove "nan" values that may have been converted to strings
cleaned_text = ' '.join([word for word in merged_text.split() if word.lower() != "nan"])

# Convert to DataFrame with a single cell
cv = pd.DataFrame({'resume': [cleaned_text]})
cv

,resume
0,TAN HAO WEN A I & DATA Engineering CONTACT Pho...


In [ ]:
def remove_numbers_with_special_characters(text):
    text = re.sub(r'\d+\.\s*',' ', text)  # Remove numbers followed by a dot and space (e.g., "1.MC")
    text = re.sub(r'\d+\)', ' ', text)  # Remove numbers followed by ' ')
    text = re.sub(r'\b\w*[a-zA-Z]+\d+[a-zA-Z]*\b', ' ', text)# Remove numbers followed by random characters and numbers examples (e.g., "abc123","a1b2c")
    text = re.sub(r'[^\w\s]', ' ', text)  # Remove other special characters
    return text


In [ ]:
cv['Process_text'] = cv['resume'].apply(remove_special_characters)
cv

,resume,Process_text
0,TAN HAO WEN A I & DATA Engineering CONTACT Pho...,TAN HAO WEN A I DATA Engineering CONTACT Phone...


In [ ]:
def remove_stopwords(text):
    stop_words = set(stopwords.words('english'))
    words = word_tokenize(text)
    filtered_words = [word for word in words if word.lower() not in stop_words]
    return text

In [ ]:
cv['Process_text'] = cv['Process_text'].apply(remove_special_characters)
cv

,resume,Process_text
0,TAN HAO WEN A I & DATA Engineering CONTACT Pho...,TAN HAO WEN A I DATA Engineering CONTACT Phone...


In [ ]:
# Drop the 'Resume'
cv = cv.drop(columns= ['resume'])

cv


,Process_text
0,TAN HAO WEN A I DATA Engineering CONTACT Phone...


In [ ]:
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
text = cv['Process_text'][0]
tokens_cv = tokenizer.tokenize(text)

tokens_cv

['T',
 'AN',
 'ĠHA',
 'O',
 'ĠW',
 'EN',
 'ĠA',
 'ĠI',
 'ĠDATA',
 'ĠEngineering',
 'ĠCONT',
 'ACT',
 'ĠPhone',
 'Ġ89',
 '33',
 'Ġ58',
 '46',
 'ĠEmail',
 'Ġh',
 'ha',
 'ow',
 'w',
 'eng',
 'mail',
 'com',
 'ĠKEY',
 'ĠSK',
 'ILL',
 'S',
 'ĠData',
 'ĠPrepar',
 'ation',
 'ĠVisual',
 'ization',
 'ĠProgramming',
 'ĠLanguages',
 'ĠMachine',
 'ĠLearning',
 'ĠModels',
 'ĠCommunication',
 'ĠTeam',
 'work',
 'Ġproblems',
 'olving',
 'ĠED',
 'UC',
 'ATION',
 'ĠN',
 'any',
 'ang',
 'ĠPoly',
 'techn',
 'ic',
 'Ġ20',
 '23',
 '20',
 '26',
 'ĠAI',
 'ĠDATA',
 'ĠENG',
 'INE',
 'ER',
 'ING',
 'ĠIT',
 'E',
 'ĠCollege',
 'ĠWest',
 'Ġ202',
 '120',
 '23',
 'ĠHigher',
 'ĠN',
 'ite',
 'c',
 'ĠMe',
 'chn',
 'ical',
 'ĠEngineering',
 'ĠCanberra',
 'ĠSecondary',
 'ĠSchool',
 'Ġ2016',
 '2020',
 'ĠG',
 'EC',
 'ĠN',
 'Ġlevel',
 'ĠEX',
 'PER',
 'IENCE',
 'ĠQ',
 'SF',
 'ĠThe',
 'ĠEn',
 'ab',
 'lers',
 'ĠP',
 'TE',
 'ĠLTD',
 'ĠCasual',
 'ĠAssistant',
 'ĠProgramme',
 'ĠStaff',
 'ĠMar',
 'Ġ2024',
 'ĠPresent',
 'ĠCult',
 '

#### One Hot Vectors###

In [ ]:

# Flatten token lists if necessary
all_tokens = [token for sublist in tokens for token in sublist]
all_tokens_cv = [token for sublist in tokens_cv for token in sublist]

# Create a shared vocabulary (unique tokens from dataset & CV)
vocabulary = list(set(all_tokens + all_tokens_cv))

# Create LabelBinarizer object and fit it to the shared vocabulary
label_binarizer = LabelBinarizer()
label_binarizer.fit(vocabulary)
# Encode dataset resumes using the shared vocabulary
one_hot_dataset = [label_binarizer.transform([token]) for token in all_tokens]

# Encode CV using the shared vocabulary
one_hot_cv = label_binarizer.transform([[token] for token in all_tokens_cv])

# Convert each resume’s One-Hot Encoded matrix into a single summed vector
one_hot_dataset_vectors = [np.sum(vec, axis=0) for vec in one_hot_dataset]  # Sum along axis
one_hot_cv_vector = np.sum(one_hot_cv, axis=0)  # Sum the vector

# Function to calculate Cosine Similarity
def calculate_cosine_similarity(doc_vector1, doc_vector2):
    return cosine_similarity(doc_vector1.reshape(1, -1), doc_vector2.reshape(1, -1))[0][0]

# Compute Cosine Similarity between CV and each resume
similarity_scores = [calculate_cosine_similarity(one_hot_cv_vector, vec) for vec in one_hot_dataset_vectors]

# Rank resumes by similarity score
top_10_indices = np.argsort(similarity_scores)[-10:][::-1]  # Get top 10 highest similarity scores

# Retrieve the corresponding resumes
top_10_resumes = [f"Resume {i+1}" for i in top_10_indices]

# Display results
top_10_df = pd.DataFrame({"Rank": range(1, 11), "Resume": top_10_resumes, "Similarity Score": np.array(similarity_scores)[top_10_indices]})
print("\nTop 10 Most Similar Resumes to CV(One Hot Vector):")
print(top_10_df)


Top 10 Most Similar Resumes to CV(One Hot Vector):
   Rank      Resume  Similarity Score
0     1   Resume 10          0.579402
1     2   Resume 21          0.579402
2     3   Resume 18          0.579402
3     4   Resume 28          0.579402
4     5  Resume 155          0.579402
5     6   Resume 39          0.579402
6     7   Resume 42          0.579402
7     8   Resume 48          0.579402
8     9   Resume 59          0.579402
9    10   Resume 56          0.579402


#### Bag of Words

In [ ]:
# Load GPT-2 Tokenizer
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

# Ensure 'df' contains 'Category' and 'Cleaned_text'
df = df[['Category', 'Cleaned_text']].dropna()

# Tokenize resumes using GPT-2 subword tokenization
df['Tokenized_text'] = df['Cleaned_text'].apply(lambda text: tokenizer.tokenize(text))

# Tokenize CV text
cv_tokens = tokenizer.tokenize(cv['Process_text'].iloc[0])

# Convert tokenized lists back to strings for CountVectorizer
df['Processed_text'] = df['Tokenized_text'].apply(lambda tokens: ' '.join(tokens))
cv_text_processed = ' '.join(cv_tokens)

# Combine all resumes and CV into a single corpus
corpus = df['Processed_text'].tolist() + [cv_text_processed]

# Initialize CountVectorizer (BoW)
vectorizer = CountVectorizer(lowercase=False)  # Keep casing for subwords

# Fit and transform the corpus into a BoW matrix
bow_matrix = vectorizer.fit_transform(corpus)

# Convert BoW matrix to dense format
bow_matrix_dense = bow_matrix.toarray()

# Convert BoW matrix to DataFrame for visualization
bow_df = pd.DataFrame(
    bow_matrix_dense,
    index=[f"Resume {i+1}" for i in range(len(df))] + ["CV"],
    columns=vectorizer.get_feature_names_out()
)

# Extract the CV vector (last row) and resume vectors
cv_vector = bow_matrix_dense[-1]  # Candidate's CV
resume_vectors = bow_matrix_dense[:-1]  # Resumes

# Compute cosine similarity between CV and all resumes
similarity_scores = cosine_similarity(resume_vectors, cv_vector.reshape(1, -1)).flatten()

# Rank the top 10 most similar resumes
top_10_indices = np.argsort(similarity_scores)[-10:][::-1]

# Retrieve the top 10 resumes
top_10_resumes = df.iloc[top_10_indices]

# Create a DataFrame for the top-ranked resumes
top_10_df = pd.DataFrame({
    "Rank": range(1, 11),
    "Category": top_10_resumes["Category"].values,
    "Resume_Index": top_10_indices + 1,  # Adjust for 1-based index
    "Similarity Score": np.array(similarity_scores)[top_10_indices]
})

# Display the Term-Document Matrix
print("\nBag-of-Words (BoW) Matrix:")
print(bow_df.T)  # Transposed for readability

# Display the top-ranked resumes
print("\nTop 10 Most Similar Resumes to CV (BoW & Cosine Similarity):")
print(top_10_df)




Bag-of-Words (BoW) Matrix:
        Resume 1  Resume 2  Resume 3  Resume 4  Resume 5  Resume 6  Resume 7  \
00             0         0         0         0         0         0         0   
001            0         0         0         0         0         0         0   
014            0         0         0         0         0         0         0   
10             0         0         0         0         0         0         0   
120            0         0         0         0         0         0         0   
...          ...       ...       ...       ...       ...       ...       ...   
Ġyou           0         0         0         0         0         0         0   
Ġyour          0         0         0         0         0         0         0   
Ġyours         0         0         0         0         1         0         0   
Ġzero          0         0         0         0         0         0         0   
Ġzone          0         0         0         0         0         0         0   

        Res

####TDM

In [ ]:
# Load GPT-2 Tokenizer
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

# Ensure 'df' contains 'Category' and 'Cleaned_text'
df = df[['Category', 'Cleaned_text']].dropna()

# Tokenize resumes using GPT-2 subword tokenization
df['Tokenized_text'] = df['Cleaned_text'].apply(lambda text: tokenizer.tokenize(text))

# Tokenize CV text
cv_tokens = tokenizer.tokenize(cv['Process_text'].iloc[0])

# Convert tokenized lists back to strings for CountVectorizer (TDM)
df['Processed_text'] = df['Tokenized_text'].apply(lambda tokens: ' '.join(tokens))
cv_text_processed = ' '.join(cv_tokens)

# Combine resumes and CV into a corpus
corpus = df['Processed_text'].tolist() + [cv_text_processed]

# Initialize CountVectorizer for Term-Document Matrix
vectorizer = CountVectorizer(lowercase=False)  # Keep casing for subwords

# Fit and transform the corpus into a term-document matrix (TDM)
td_matrix = vectorizer.fit_transform(corpus)

# Convert TDM to DataFrame
tdm = pd.DataFrame(
    td_matrix.toarray(),
    index=[f"Document {i+1}" for i in range(len(corpus))],  # Label documents
    columns=vectorizer.get_feature_names_out()
)

# Transpose for better visualization
tdm_transposed = tdm.T

# Extract the CV vector (last document)
cv_vector = tdm.iloc[-1].values.reshape(1, -1)

# Extract resume vectors (all rows except the last one)
resume_vectors = tdm.iloc[:-1].values

# Compute similarity scores using cosine similarity
similarity_scores = cosine_similarity(resume_vectors, cv_vector).flatten()

# Get the top 10 most similar resumes
top_10_indices = np.argsort(similarity_scores)[-10:][::-1]  # Get top 10 highest scores

# Retrieve the corresponding resumes and categories
top_10_resumes = df.iloc[top_10_indices]

# Create a DataFrame for the top 10 ranked resumes
top_10_df = pd.DataFrame({
    "Rank": range(1, 11),
    "Category": top_10_resumes["Category"].values,
    "Resume_Index": top_10_indices + 1,  # Adjust for 1-based index
    "Similarity Score": np.array(similarity_scores)[top_10_indices]
})

# Display results
print("\nTop 10 Most Similar Resumes to CV(TDM):")
print(top_10_df)




Top 10 Most Similar Resumes to CV(TDM):
   Rank             Category  Resume_Index  Similarity Score
0     1                  PMO            57          0.543403
1     2                 Arts            74          0.525932
2     3  Mechanical Engineer            47          0.512556
3     4   Operations Manager            10          0.511874
4     5   Operations Manager            36          0.502220
5     6     Business Analyst            66          0.501743
6     7              Testing           120          0.494938
7     8   Automation Testing            88          0.490470
8     9      DevOps Engineer            20          0.489763
9    10   Operations Manager            48          0.475368


#### TF-IDF

In [ ]:
# Function to calculate Term Frequency (TF)
def calculate_tf(value):
    return np.log10(value + 1)

# Function to calculate Inverse Document Frequency (IDF)
def calculate_idf(total_documents, value):
    return np.log10(total_documents / value)

# Total number of documents (including CV)
total_documents = len(corpus)

# Number of documents each term appears in (IDF calculation)
total_documents_per_word = np.sum(tdm > 0, axis=0)

# Compute IDF for each term
idf_array = total_documents_per_word.apply(lambda value: calculate_idf(total_documents, value))

# Compute TF matrix
tf_matrix = tdm.map(calculate_tf)

# Compute TF-IDF matrix
tf_idf_matrix = tf_matrix * idf_array.to_numpy()


# Extract the CV vector (last row)
cv_vector = tf_idf_matrix.iloc[-1].values.reshape(1, -1)

# Extract resume vectors (all rows except the last one)
resume_vectors = tf_idf_matrix.iloc[:-1].values

# Compute cosine similarity between CV and all resumes
similarity_scores = cosine_similarity(resume_vectors, cv_vector).flatten()

# Rank the top 10 most similar resumes
top_10_indices = np.argsort(similarity_scores)[-10:][::-1]

# Retrieve the top 10 resumes
top_10_resumes = df.iloc[top_10_indices]

# Create a DataFrame for the top-ranked resumes
top_10_df = pd.DataFrame({
    "Rank": range(1, 11),
    "Category": top_10_resumes["Category"].values,
    "Resume_Index": top_10_indices + 1,  # Adjust for 1-based index
    "Similarity Score": np.array(similarity_scores)[top_10_indices]
})
# Display results
print("\nTop 10 Most Similar Resumes to CV (TF-IDF & Cosine Similarity):")
print(top_10_df)



Top 10 Most Similar Resumes to CV (TF-IDF & Cosine Similarity):
   Rank             Category  Resume_Index  Similarity Score
0     1   Operations Manager            48          0.071209
1     2   Operations Manager            36          0.068815
2     3                 Arts            74          0.064816
3     4         Data Science           176          0.062040
4     5  Mechanical Engineer            47          0.061395
5     6     Business Analyst            66          0.060427
6     7        Web Designing            90          0.059629
7     8   Operations Manager           113          0.054624
8     9                  PMO            57          0.054492
9    10             Database           172          0.054395


#### Word2vec


In [ ]:
# Load the saved Word2Vec model
word2vec_model = gensim.downloader.load('glove-wiki-gigaword-50')

# Function to tokenize and vectorize documents using Word2Vec
def vectorize_documents(documents, model):
    document_vectors = []
    for document in documents:
        tokens = document.lower().split()
        vectors = [model[token] for token in tokens if token in model]
        if vectors:
            document_vectors.append(sum(vectors) / len(vectors))  # Average Word Embeddings
        else:
            document_vectors.append(np.zeros(50))  # If no known words, use zero vector
    return np.array(document_vectors)

# Convert documents into vectors
document_vectors = vectorize_documents(corpus, word2vec_model)

# Extract the CV vector (last row)
cv_vector = document_vectors[-1].reshape(1, -1)

# Extract resume vectors (all rows except the last one)
resume_vectors = document_vectors[:-1]

# Compute cosine similarity
similarity_scores = cosine_similarity(resume_vectors, cv_vector).flatten()

# Get top 10 most similar resumes
top_10_indices = np.argsort(similarity_scores)[-10:][::-1]  # Get top 10 highest similarity scores

# Retrieve the corresponding resumes and categories
top_10_resumes = df.iloc[top_10_indices]

# Create a DataFrame for the top 10 ranked resumes
top_10_df = pd.DataFrame({
    "Rank": range(1, 11),
    "Category": top_10_resumes["Category"].values,
    "Resume_Index": top_10_indices + 1,  # Adjust for 1-based index
    "Similarity Score": np.array(similarity_scores)[top_10_indices]
})

# Display results
print("\nTop 10 Most Similar Resumes to CV (Word2Vec & Cosine Similarity):")
print(top_10_df)


[==================================================] 100.0% 66.0/66.0MB downloaded

Top 10 Most Similar Resumes to CV (Word2Vec & Cosine Similarity):
   Rank             Category  Resume_Index  Similarity Score
0     1  Mechanical Engineer           135          0.929061
1     2       Civil Engineer            42          0.925255
2     3       Civil Engineer           127          0.924878
3     4                  PMO           182          0.924315
4     5       Java Developer           155          0.923922
5     6         Data Science           176          0.923198
6     7                  PMO           133          0.922440
7     8   Operations Manager            10          0.922046
8     9       Civil Engineer           110          0.921578
9    10   Operations Manager            48          0.911406


## Week 17

### Bert fine tuning ###

In [ ]:
df

,Category,Cleaned_text,Tokenized_text,Processed_text
0,HR,Education Details BA mumbai University HR Skil...,"[Education, ĠDetails, ĠBA, Ġm, umbai, ĠUnivers...",Education ĠDetails ĠBA Ġm umbai ĠUniversity ĠH...
1,Electrical Engineering,Skills MC Office AutoCAD 2016 Introductory Kno...,"[Sk, ills, ĠMC, ĠOffice, ĠAuto, C, AD, Ġ2016, ...",Sk ills ĠMC ĠOffice ĠAuto C AD Ġ2016 ĠIntrodu ...
2,Advocate,Skills Legal Writing Efficient researcher Lega...,"[Sk, ills, ĠLegal, ĠWriting, ĠE, fficient, Ġre...",Sk ills ĠLegal ĠWriting ĠE fficient Ġresearche...
4,Testing,Computer Skills Proficient in MS office Word B...,"[Computer, ĠSkills, ĠProf, icient, Ġin, ĠMS, Ġ...",Computer ĠSkills ĠProf icient Ġin ĠMS Ġoffice ...
5,ETL Developer,Computer skills Yes SQL knowledge yes Unix kno...,"[Computer, Ġskills, ĠYes, ĠSQL, Ġknowledge, Ġy...",Computer Ġskills ĠYes ĠSQL Ġknowledge Ġyes ĠUn...
...,...,...,...,...
763,SAP Developer,Skills ETL Data Warehousing SQL PL SQL Basic C...,"[Sk, ills, ĠET, L, ĠData, ĠWare, housing, ĠSQL...",Sk ills ĠET L ĠData ĠWare housing ĠSQL ĠPL ĠSQ...
800,Sales,Skill Sets Multi tasking Collaborative Optimis...,"[Skill, ĠSets, ĠMulti, Ġtask, ing, ĠCollabor, ...",Skill ĠSets ĠMulti Ġtask ing ĠCollabor ative Ġ...
850,PMO,AREA OF EXPERTISE PROFILE Around 10 plus years...,"[ARE, A, ĠOF, ĠEXP, ERT, ISE, ĠPRO, FILE, ĠAro...",ARE A ĠOF ĠEXP ERT ISE ĠPRO FILE ĠAround Ġ10 Ġ...
870,DevOps Engineer,Technical Skills Key Skills MS Technology Net ...,"[Technical, ĠSkills, ĠKey, ĠSkills, ĠMS, ĠTech...",Technical ĠSkills ĠKey ĠSkills ĠMS ĠTechnology...


In [ ]:
df.Cleaned_text

0      Education Details BA mumbai University HR Skil...
1      Skills MC Office AutoCAD 2016 Introductory Kno...
2      Skills Legal Writing Efficient researcher Lega...
4      Computer Skills Proficient in MS office Word B...
5      Computer skills Yes SQL knowledge yes Unix kno...
                             ...                        
763    Skills ETL Data Warehousing SQL PL SQL Basic C...
800    Skill Sets Multi tasking Collaborative Optimis...
850    AREA OF EXPERTISE PROFILE Around 10 plus years...
870    Technical Skills Key Skills MS Technology Net ...
900    TECHNICAL SKILLS Programming Languages C NET W...
Name: Cleaned_text, Length: 184, dtype: object

In [ ]:
num_classes = df["Category"].nunique()
print(f"Number of Categories: {num_classes}\n")

Number of Categories: 25



In [ ]:
resumes=df['Cleaned_text'].values
sentences = ["[CLS] " + resume + " [SEP]" for resume in resumes]
labels = df['Category'].values

In [ ]:
# Generate tokens and input IDs for training
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased', do_lower_case=True)
tokenized_sentences = [tokenizer.tokenize(sentence) for sentence in sentences]

input_ids = [tokenizer.convert_tokens_to_ids(tokenized_sentence) for tokenized_sentence in tokenized_sentences ]
input_ids = pad_sequences(input_ids, maxlen=512, dtype="long", truncating="post", padding="post")
input_ids.shape

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

(184, 512)

In [ ]:
# Generate attention masks for training
attention_masks = [[float(i > 0) for i in input_id] for input_id in input_ids]

In [ ]:
# Split data into training and validation sets, use straified split due to class imbalance
training_inputs, validation_inputs, training_labels, validation_labels, training_masks,validation_masks = train_test_split(input_ids,labels, attention_masks, random_state=2018, test_size=0.15, stratify=labels)


# # Check the sizes of each set
# Check the sizes of each set
print("Training Set Size:", len(training_inputs))
print("Validation Set Size:", len(validation_inputs))


Training Set Size: 156
Validation Set Size: 28


In [ ]:
print(type(training_inputs))
print(len(training_inputs))  # Number of samples
print([len(x) for x in training_inputs[:10]])  # Length of the first 10 samples


<class 'numpy.ndarray'>
156
[512, 512, 512, 512, 512, 512, 512, 512, 512, 512]


In [ ]:
print([len(seq) for seq in training_inputs])


[512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512]


In [ ]:
print(type(training_inputs))  # Should be <class 'list'>
print(type(training_inputs[0]))  # Should be <class 'list'> or <class 'np.ndarray'>

<class 'numpy.ndarray'>
<class 'numpy.ndarray'>


In [ ]:
batch_size=32
from sklearn.preprocessing import LabelEncoder

# Initialize LabelEncoder
label_encoder = LabelEncoder()

# Fit and transform the training labels
training_labels = label_encoder.fit_transform(training_labels)
validation_labels = label_encoder.transform(validation_labels)  # Use transform to maintain consistency

# Convert to numpy arrays of int64
training_labels = np.array(training_labels, dtype=np.int64)
validation_labels = np.array(validation_labels, dtype=np.int64)

# Ensure correct data types before converting to tensors
training_inputs = np.array(training_inputs, dtype=np.int64)  # Convert to int64
training_masks = np.array(training_masks, dtype=np.int64)      # Convert to int64
training_labels = np.array(training_labels, dtype=np.int64)    # Convert to int64

validation_inputs = np.array(validation_inputs, dtype=np.int64)  # Convert to int64
validation_masks = np.array(validation_masks, dtype=np.int64)      # Convert to int64
validation_labels = np.array(validation_labels, dtype=np.int64)    # Convert to int64

# Convert to PyTorch tensors
training_inputs = TensorDataset(
    torch.tensor(training_inputs, dtype=torch.long),
    torch.tensor(training_masks, dtype=torch.long),
    torch.tensor(training_labels, dtype=torch.long)
)

training_sampler = RandomSampler(training_inputs)
training_dataloader = DataLoader(training_inputs, sampler=training_sampler, batch_size=batch_size)

validation_inputs = TensorDataset(
    torch.tensor(validation_inputs, dtype=torch.long),
    torch.tensor(validation_masks, dtype=torch.long),
    torch.tensor(validation_labels, dtype=torch.long)
)

validation_sampler = SequentialSampler(validation_inputs)
validation_dataloader = DataLoader(validation_inputs, sampler=validation_sampler, batch_size=batch_size)

In [ ]:
# Prepare model
configuration = BertConfig()
model = BertModel(configuration)
config = model.config

model = BertForSequenceClassification.from_pretrained("bert-base-uncased",num_labels=25)


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
model = nn.DataParallel(model)
model.to(device)

DataParallel(
  (module): BertForSequenceClassification(
    (bert): BertModel(
      (embeddings): BertEmbeddings(
        (word_embeddings): Embedding(30522, 768, padding_idx=0)
        (position_embeddings): Embedding(512, 768)
        (token_type_embeddings): Embedding(2, 768)
        (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (encoder): BertEncoder(
        (layer): ModuleList(
          (0-11): 12 x BertLayer(
            (attention): BertAttention(
              (self): BertSdpaSelfAttention(
                (query): Linear(in_features=768, out_features=768, bias=True)
                (key): Linear(in_features=768, out_features=768, bias=True)
                (value): Linear(in_features=768, out_features=768, bias=True)
                (dropout): Dropout(p=0.1, inplace=False)
              )
              (output): BertSelfOutput(
                (dense): Linear(in_features=768, out_features=7

In [ ]:
# Prepare AdamW optimizer
parameters = list(model.named_parameters())
no_decay = ['bias', 'LayerNorm.weight']
parameters = [
{'params': [p for n, p in parameters if not any(nd in n for nd in no_decay)], 'weight_decay':
0.01},
{'params': [p for n, p in parameters if any(nd in n for nd in no_decay)], 'weight_decay': 0.0}
]
optimizer = AdamW(parameters, lr=2e-5, correct_bias=False)

/usr/local/lib/python3.10/dist-packages/transformers/optimization.py:640: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


In [ ]:
# Calculate accuracy
def accuracy(predicted_labels, labels):
  predicted_labels = numpy.argmax(predicted_labels.to('cpu').numpy(), axis=1).flatten()
  labels = labels.to('cpu').numpy().flatten()
  return numpy.sum(predicted_labels == labels) / len(labels)

In [ ]:
# Train model
epochs = 50
training_losses = []

for epoch in trange(epochs, desc="Epoch"):
    model.train()
    training_loss = 0
    training_steps = 0

    for step, batch in enumerate(training_dataloader):  # Indentation fixed
        inputs = batch[0].to(device)
        attention_masks = batch[1].to(device)
        labels = batch[2].to(device)

        optimizer.zero_grad()
        outputs = model(inputs, attention_mask=attention_masks, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()

        training_loss += loss.item()
        training_steps += 1
        training_losses.append(loss.item())

    average_training_loss = training_loss / training_steps
    print("Epoch {}: Average Training Loss: {}".format(epoch + 1, average_training_loss))

    # Validation
    model.eval()
    validation_accuracy = 0
    validation_steps = 0

    for batch in validation_dataloader:
        inputs = batch[0].to(device)
        attention_masks = batch[1].to(device)
        labels = batch[2].to(device)

        with torch.no_grad():
            outputs = model(inputs, attention_mask=attention_masks, labels=labels)
        logits = outputs.logits

        temp_validation_accuracy = accuracy(logits, labels)
        validation_accuracy += temp_validation_accuracy
        validation_steps += 1

    average_validation_accuracy = validation_accuracy / validation_steps
    print("Epoch {}: Validation Accuracy: {}".format(epoch + 1, average_validation_accuracy))


Epoch:   0%|          | 0/50 [00:00<?, ?it/s]

Epoch 1: Average Training Loss: 3.2844173431396486


Epoch:   2%|▏         | 1/50 [00:08<07:20,  8.99s/it]

Epoch 1: Validation Accuracy: 0.07142857142857142
Epoch 2: Average Training Loss: 3.136320638656616


Epoch:   4%|▍         | 2/50 [00:17<06:56,  8.67s/it]

...

Epoch 48: Validation Accuracy: 0.9285714285714286
Epoch 49: Average Training Loss: 0.047151331603527066


Epoch:  98%|█████████▊| 49/50 [06:58<00:08,  8.54s/it]

Epoch 49: Validation Accuracy: 0.9285714285714286
Epoch 50: Average Training Loss: 0.045353783667087554


Epoch: 100%|██████████| 50/50 [07:07<00:00,  8.54s/it]

Epoch 50: Validation Accuracy: 0.9285714285714286


In [ ]:
resumes=df['Cleaned_text'].values
sentences = ["[CLS] " + resume + " [SEP]" for resume in resumes]
labels = df['Category'].values

In [ ]:
# Generate tokens and input IDs for evaluation
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased', do_lower_case=True)
tokenized_sentences = [tokenizer.tokenize(sentence) for sentence in sentences]
input_ids = [tokenizer.convert_tokens_to_ids(tokenized_sentence) for tokenized_sentence in tokenized_sentences]
input_ids = pad_sequences(input_ids, maxlen=512, dtype="long",truncating="post", padding="post")
input_ids = torch.tensor(input_ids)

In [ ]:
# Generate attention masks for evaluation
attention_masks = [[float(i > 0) for i in input_id] for input_id in input_ids]
attention_masks = torch.tensor(attention_masks)


In [ ]:
print(type(input_ids), type(attention_masks), type(labels))
print(input_ids.shape, attention_masks.shape, labels.shape)


<class 'torch.Tensor'> <class 'torch.Tensor'> <class 'numpy.ndarray'>
torch.Size([184, 512]) torch.Size([184, 512]) (184,)


In [ ]:
labels = label_encoder.fit_transform(labels)
labels = np.array(labels, dtype=np.int64)
labels = torch.tensor(labels, dtype=torch.long)

In [ ]:
# Prepare prediction dataset
prediction_dataset = TensorDataset(input_ids, attention_masks, labels)
prediction_dataloader = DataLoader(prediction_dataset, batch_size=batch_size)

# Evaluate model
model.eval()
logits_set = []
labels_set = []

for batch in prediction_dataloader:
    batch_input_ids, batch_attention_masks, batch_labels = batch  # Unpack correctly
    batch_input_ids = batch_input_ids.to(device)
    batch_attention_masks = batch_attention_masks.to(device)
    batch_labels = batch_labels.to(device)

    with torch.no_grad():
        outputs = model(batch_input_ids, attention_mask=batch_attention_masks)
        logits = outputs.logits

    logits_set.append(logits.cpu().numpy())
    labels_set.append(batch_labels.cpu().numpy())


In [ ]:
# Calculate Matthews correlation coefficient for each batch
matthews_set = []
for i in range(len(labels_set)):
  mcc = matthews_corrcoef(labels_set[i], numpy.argmax(logits_set[i], axis=1).flatten())
  matthews_set.append(mcc)
for i, mcc in enumerate(matthews_set):
  print(f"Batch {i + 1}: MCC = {mcc}")
# Calculate the overall Matthews correlation coefficient
overall_mcc = numpy.mean(matthews_set)
print(f"\nOverall MCC: {overall_mcc}")

Batch 1: MCC = 1.0
Batch 2: MCC = 1.0
Batch 3: MCC = 1.0
Batch 4: MCC = 1.0
Batch 5: MCC = 1.0
Batch 6: MCC = 0.9118840877808192

Overall MCC: 0.9853140146301366


## Week 18

### Optimize

In [ ]:
df

,Category,Cleaned_text,Tokenized_text,Processed_text
0,HR,Education Details BA mumbai University HR Skil...,"[Education, ĠDetails, ĠBA, Ġm, umbai, ĠUnivers...",Education ĠDetails ĠBA Ġm umbai ĠUniversity ĠH...
1,Electrical Engineering,Skills MC Office AutoCAD 2016 Introductory Kno...,"[Sk, ills, ĠMC, ĠOffice, ĠAuto, C, AD, Ġ2016, ...",Sk ills ĠMC ĠOffice ĠAuto C AD Ġ2016 ĠIntrodu ...
2,Advocate,Skills Legal Writing Efficient researcher Lega...,"[Sk, ills, ĠLegal, ĠWriting, ĠE, fficient, Ġre...",Sk ills ĠLegal ĠWriting ĠE fficient Ġresearche...
4,Testing,Computer Skills Proficient in MS office Word B...,"[Computer, ĠSkills, ĠProf, icient, Ġin, ĠMS, Ġ...",Computer ĠSkills ĠProf icient Ġin ĠMS Ġoffice ...
5,ETL Developer,Computer skills Yes SQL knowledge yes Unix kno...,"[Computer, Ġskills, ĠYes, ĠSQL, Ġknowledge, Ġy...",Computer Ġskills ĠYes ĠSQL Ġknowledge Ġyes ĠUn...
...,...,...,...,...
763,SAP Developer,Skills ETL Data Warehousing SQL PL SQL Basic C...,"[Sk, ills, ĠET, L, ĠData, ĠWare, housing, ĠSQL...",Sk ills ĠET L ĠData ĠWare housing ĠSQL ĠPL ĠSQ...
800,Sales,Skill Sets Multi tasking Collaborative Optimis...,"[Skill, ĠSets, ĠMulti, Ġtask, ing, ĠCollabor, ...",Skill ĠSets ĠMulti Ġtask ing ĠCollabor ative Ġ...
850,PMO,AREA OF EXPERTISE PROFILE Around 10 plus years...,"[ARE, A, ĠOF, ĠEXP, ERT, ISE, ĠPRO, FILE, ĠAro...",ARE A ĠOF ĠEXP ERT ISE ĠPRO FILE ĠAround Ġ10 Ġ...
870,DevOps Engineer,Technical Skills Key Skills MS Technology Net ...,"[Technical, ĠSkills, ĠKey, ĠSkills, ĠMS, ĠTech...",Technical ĠSkills ĠKey ĠSkills ĠMS ĠTechnology...


In [ ]:
df.Cleaned_text

0      Education Details BA mumbai University HR Skil...
1      Skills MC Office AutoCAD 2016 Introductory Kno...
2      Skills Legal Writing Efficient researcher Lega...
4      Computer Skills Proficient in MS office Word B...
5      Computer skills Yes SQL knowledge yes Unix kno...
                             ...                        
763    Skills ETL Data Warehousing SQL PL SQL Basic C...
800    Skill Sets Multi tasking Collaborative Optimis...
850    AREA OF EXPERTISE PROFILE Around 10 plus years...
870    Technical Skills Key Skills MS Technology Net ...
900    TECHNICAL SKILLS Programming Languages C NET W...
Name: Cleaned_text, Length: 184, dtype: object

In [ ]:
num_classes = df["Category"].nunique()
print(f"Number of Categories: {num_classes}\n")

Number of Categories: 25



In [ ]:
resumes=df['Cleaned_text'].values
sentences = ["[CLS] " + resume + " [SEP]" for resume in resumes]
labels = df['Category'].values

In [ ]:
# Generate tokens and input IDs for training
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased', do_lower_case=True)
tokenized_sentences = [tokenizer.tokenize(sentence) for sentence in sentences]

input_ids = [tokenizer.convert_tokens_to_ids(tokenized_sentence) for tokenized_sentence in tokenized_sentences ]
input_ids = pad_sequences(input_ids, maxlen=512, dtype="long", truncating="post", padding="post")
input_ids.shape

(184, 512)

In [ ]:
# Generate attention masks for training
attention_masks = [[float(i > 0) for i in input_id] for input_id in input_ids]

In [ ]:
# Split data into training and validation sets, use straified split due to class imbalance
training_inputs, validation_inputs, training_labels, validation_labels, training_masks,validation_masks = train_test_split(input_ids,labels, attention_masks, random_state=2018, test_size=0.15, stratify=labels)



# # Check the sizes of each set
# Check the sizes of each set
print("Training Set Size:", len(training_inputs))
print("Validation Set Size:", len(validation_inputs))


Training Set Size: 156
Validation Set Size: 28


In [ ]:
print(type(training_inputs))
print(len(training_inputs))  # Number of samples
print([len(x) for x in training_inputs[:10]])  # Length of the first 10 samples


<class 'numpy.ndarray'>
156
[512, 512, 512, 512, 512, 512, 512, 512, 512, 512]


In [ ]:
print([len(seq) for seq in training_inputs])


[512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512]


In [ ]:
print(type(training_inputs))  # Should be <class 'list'>
print(type(training_inputs[0]))  # Should be <class 'list'> or <class 'np.ndarray'>

<class 'numpy.ndarray'>
<class 'numpy.ndarray'>


In [ ]:
batch_size=32
from sklearn.preprocessing import LabelEncoder

# Initialize LabelEncoder
label_encoder = LabelEncoder()

# Fit and transform the training labels
training_labels = label_encoder.fit_transform(training_labels)
validation_labels = label_encoder.transform(validation_labels)  # Use transform to maintain consistency

# Convert to numpy arrays of int64
training_labels = np.array(training_labels, dtype=np.int64)
validation_labels = np.array(validation_labels, dtype=np.int64)

# Ensure correct data types before converting to tensors
training_inputs = np.array(training_inputs, dtype=np.int64)  # Convert to int64
training_masks = np.array(training_masks, dtype=np.int64)      # Convert to int64
training_labels = np.array(training_labels, dtype=np.int64)    # Convert to int64

validation_inputs = np.array(validation_inputs, dtype=np.int64)  # Convert to int64
validation_masks = np.array(validation_masks, dtype=np.int64)      # Convert to int64
validation_labels = np.array(validation_labels, dtype=np.int64)    # Convert to int64

# Convert to PyTorch tensors
training_inputs = TensorDataset(
    torch.tensor(training_inputs, dtype=torch.long),
    torch.tensor(training_masks, dtype=torch.long),
    torch.tensor(training_labels, dtype=torch.long)
)

training_sampler = RandomSampler(training_inputs)
training_dataloader = DataLoader(training_inputs, sampler=training_sampler, batch_size=batch_size)

validation_inputs = TensorDataset(
    torch.tensor(validation_inputs, dtype=torch.long),
    torch.tensor(validation_masks, dtype=torch.long),
    torch.tensor(validation_labels, dtype=torch.long)
)

validation_sampler = SequentialSampler(validation_inputs)
validation_dataloader = DataLoader(validation_inputs, sampler=validation_sampler, batch_size=batch_size)

In [ ]:
# Prepare model
configuration = BertConfig()
model = BertModel(configuration)
config = model.config

model = BertForSequenceClassification.from_pretrained("bert-base-uncased",num_labels=25)
model = nn.DataParallel(model)
model.to(device)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


DataParallel(
  (module): BertForSequenceClassification(
    (bert): BertModel(
      (embeddings): BertEmbeddings(
        (word_embeddings): Embedding(30522, 768, padding_idx=0)
        (position_embeddings): Embedding(512, 768)
        (token_type_embeddings): Embedding(2, 768)
        (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (encoder): BertEncoder(
        (layer): ModuleList(
          (0-11): 12 x BertLayer(
            (attention): BertAttention(
              (self): BertSdpaSelfAttention(
                (query): Linear(in_features=768, out_features=768, bias=True)
                (key): Linear(in_features=768, out_features=768, bias=True)
                (value): Linear(in_features=768, out_features=768, bias=True)
                (dropout): Dropout(p=0.1, inplace=False)
              )
              (output): BertSelfOutput(
                (dense): Linear(in_features=768, out_features=7

In [ ]:
# Prepare AdamW optimizer
parameters = list(model.named_parameters())
no_decay = ['bias', 'LayerNorm.weight']
parameters = [
{'params': [p for n, p in parameters if not any(nd in n for nd in no_decay)], 'weight_decay':
0.01},
{'params': [p for n, p in parameters if any(nd in n for nd in no_decay)], 'weight_decay': 0.0}
]
optimizer = AdamW(parameters, lr=0.00005, correct_bias=False)

/usr/local/lib/python3.10/dist-packages/transformers/optimization.py:640: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


In [ ]:
epochs = 50
# Compute class weights for imbalance
class_weights = compute_class_weight(class_weight="balanced", classes=np.unique(training_labels), y=training_labels)
class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)
loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights)

# Define optimizer and scheduler
# optimizer = AdamW(model.parameters(), lr=2e-5, betas=(0.9, 0.98), eps=1e-8, weight_decay=0.01)
num_training_steps = epochs * len(training_dataloader)
lr_scheduler = get_scheduler("linear", optimizer=optimizer, num_warmup_steps=100, num_training_steps=num_training_steps)

# Early Stopping Parameters
early_stopping_patience = 5
best_val_accuracy = 0
epochs_no_improve = 0

In [ ]:
# Calculate accuracy
def accuracy(predicted_labels, labels):
  predicted_labels = numpy.argmax(predicted_labels.to('cpu').numpy(), axis=1).flatten()
  labels = labels.to('cpu').numpy().flatten()
  return numpy.sum(predicted_labels == labels) / len(labels)

In [ ]:
# Training Loop with Gradient Accumulation
epochs = 50
training_losses = []
accumulation_steps = 2  # Simulates batch_size=64

for epoch in trange(epochs, desc="Epoch"):
    model.train()
    training_loss = 0
    training_steps = 0

    for step, batch in enumerate(training_dataloader):
        inputs = batch[0].to(device)
        attention_masks = batch[1].to(device)
        labels = batch[2].to(device)

        # Forward pass
        outputs = model(inputs, attention_mask=attention_masks, labels=labels)
        loss = loss_fn(outputs.logits, labels) / accumulation_steps  # Apply class weighting & scale loss
        loss.backward()

        # Perform optimizer step after accumulation_steps
        if (step + 1) % accumulation_steps == 0:
            optimizer.step()
            lr_scheduler.step()
            optimizer.zero_grad()

        training_loss += loss.item()
        training_steps += 1
        training_losses.append(loss.item())

    # Compute Average Training Loss
    average_training_loss = training_loss / training_steps
    print("Epoch {}: Average Training Loss: {}".format(epoch + 1, average_training_loss))

    # Validation Step
    model.eval()
    validation_accuracy = 0
    validation_steps = 0

    for batch in validation_dataloader:
        inputs = batch[0].to(device)
        attention_masks = batch[1].to(device)
        labels = batch[2].to(device)

        with torch.no_grad():
            outputs = model(inputs, attention_mask=attention_masks, labels=labels)
        logits = outputs.logits

        temp_validation_accuracy = accuracy(logits, labels)
        validation_accuracy += temp_validation_accuracy
        validation_steps += 1

    # Compute Average Validation Accuracy
    average_validation_accuracy = validation_accuracy / validation_steps
    print("Epoch {}: Validation Accuracy: {}".format(epoch + 1, average_validation_accuracy))

    # Early Stopping Check
    if average_validation_accuracy > best_val_accuracy:
        best_val_accuracy = average_validation_accuracy
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1

    if epochs_no_improve >= early_stopping_patience:
        print("Early stopping triggered! Stopping training.")
        break



Epoch:   0%|          | 0/50 [00:00<?, ?it/s]

Epoch 1: Average Training Loss: 1.6432406902313232


Epoch:   2%|▏         | 1/50 [00:08<06:54,  8.47s/it]

Epoch 1: Validation Accuracy: 0.07142857142857142
Epoch 2: Average Training Loss: 1.633070945739746


Epoch:   4%|▍         | 2/50 [00:16<06:47,  8.49s/it]

...

Epoch 37: Validation Accuracy: 0.8214285714285714
Epoch 38: Average Training Loss: 0.0560562938451767


Epoch:  76%|███████▌  | 38/50 [05:23<01:42,  8.50s/it]

Epoch 38: Validation Accuracy: 0.8571428571428571
Epoch 39: Average Training Loss: 0.05112457573413849


Epoch:  76%|███████▌  | 38/50 [05:31<01:44,  8.73s/it]

Epoch 39: Validation Accuracy: 0.8571428571428571
Early stopping triggered! Stopping training.


In [ ]:
resumes=df['Cleaned_text'].values
sentences = ["[CLS] " + resume + " [SEP]" for resume in resumes]
labels = df['Category'].values

In [ ]:
# Generate tokens and input IDs for evaluation
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased', do_lower_case=True)
tokenized_sentences = [tokenizer.tokenize(sentence) for sentence in sentences]
input_ids = [tokenizer.convert_tokens_to_ids(tokenized_sentence) for tokenized_sentence in tokenized_sentences]
input_ids = pad_sequences(input_ids, maxlen=512, dtype="long",truncating="post", padding="post")
input_ids = torch.tensor(input_ids)

In [ ]:
# Generate attention masks for evaluation
attention_masks = [[float(i > 0) for i in input_id] for input_id in input_ids]
attention_masks = torch.tensor(attention_masks)


In [ ]:
print(type(input_ids), type(attention_masks), type(labels))
print(input_ids.shape, attention_masks.shape, labels.shape)


<class 'torch.Tensor'> <class 'torch.Tensor'> <class 'numpy.ndarray'>
torch.Size([184, 512]) torch.Size([184, 512]) (184,)


In [ ]:
labels = label_encoder.fit_transform(labels)
labels = np.array(labels, dtype=np.int64)
labels = torch.tensor(labels, dtype=torch.long)

In [ ]:
# Prepare prediction dataset
prediction_dataset = TensorDataset(input_ids, attention_masks, labels)
prediction_dataloader = DataLoader(prediction_dataset, batch_size=batch_size)

# Evaluate model
model.eval()
logits_set = []
labels_set = []

for batch in prediction_dataloader:
    batch_input_ids, batch_attention_masks, batch_labels = batch  # Unpack correctly
    batch_input_ids = batch_input_ids.to(device)
    batch_attention_masks = batch_attention_masks.to(device)
    batch_labels = batch_labels.to(device)

    with torch.no_grad():
        outputs = model(batch_input_ids, attention_mask=batch_attention_masks)
        logits = outputs.logits

    logits_set.append(logits.cpu().numpy())
    labels_set.append(batch_labels.cpu().numpy())


In [ ]:
# Calculate Matthews correlation coefficient for each batch
matthews_set = []
for i in range(len(labels_set)):
  mcc = matthews_corrcoef(labels_set[i], numpy.argmax(logits_set[i], axis=1).flatten())
  matthews_set.append(mcc)
for i, mcc in enumerate(matthews_set):
  print(f"Batch {i + 1}: MCC = {mcc}")
# Calculate the overall Matthews correlation coefficient
overall_mcc = numpy.mean(matthews_set)
print(f"\nOverall MCC: {overall_mcc}")

Batch 1: MCC = 0.9674716763828275
Batch 2: MCC = 0.9677754677754677
Batch 3: MCC = 0.934169788999084
Batch 4: MCC = 1.0
Batch 5: MCC = 1.0
Batch 6: MCC = 1.0

Overall MCC: 0.9782361555262299


### Optimize 2

In [ ]:
df

,Category,Cleaned_text,Tokenized_text,Processed_text
0,HR,Education Details BA mumbai University HR Skil...,"[Education, ĠDetails, ĠBA, Ġm, umbai, ĠUnivers...",Education ĠDetails ĠBA Ġm umbai ĠUniversity ĠH...
1,Electrical Engineering,Skills MC Office AutoCAD 2016 Introductory Kno...,"[Sk, ills, ĠMC, ĠOffice, ĠAuto, C, AD, Ġ2016, ...",Sk ills ĠMC ĠOffice ĠAuto C AD Ġ2016 ĠIntrodu ...
2,Advocate,Skills Legal Writing Efficient researcher Lega...,"[Sk, ills, ĠLegal, ĠWriting, ĠE, fficient, Ġre...",Sk ills ĠLegal ĠWriting ĠE fficient Ġresearche...
4,Testing,Computer Skills Proficient in MS office Word B...,"[Computer, ĠSkills, ĠProf, icient, Ġin, ĠMS, Ġ...",Computer ĠSkills ĠProf icient Ġin ĠMS Ġoffice ...
5,ETL Developer,Computer skills Yes SQL knowledge yes Unix kno...,"[Computer, Ġskills, ĠYes, ĠSQL, Ġknowledge, Ġy...",Computer Ġskills ĠYes ĠSQL Ġknowledge Ġyes ĠUn...
...,...,...,...,...
763,SAP Developer,Skills ETL Data Warehousing SQL PL SQL Basic C...,"[Sk, ills, ĠET, L, ĠData, ĠWare, housing, ĠSQL...",Sk ills ĠET L ĠData ĠWare housing ĠSQL ĠPL ĠSQ...
800,Sales,Skill Sets Multi tasking Collaborative Optimis...,"[Skill, ĠSets, ĠMulti, Ġtask, ing, ĠCollabor, ...",Skill ĠSets ĠMulti Ġtask ing ĠCollabor ative Ġ...
850,PMO,AREA OF EXPERTISE PROFILE Around 10 plus years...,"[ARE, A, ĠOF, ĠEXP, ERT, ISE, ĠPRO, FILE, ĠAro...",ARE A ĠOF ĠEXP ERT ISE ĠPRO FILE ĠAround Ġ10 Ġ...
870,DevOps Engineer,Technical Skills Key Skills MS Technology Net ...,"[Technical, ĠSkills, ĠKey, ĠSkills, ĠMS, ĠTech...",Technical ĠSkills ĠKey ĠSkills ĠMS ĠTechnology...


In [ ]:
df.Cleaned_text

0      Education Details BA mumbai University HR Skil...
1      Skills MC Office AutoCAD 2016 Introductory Kno...
2      Skills Legal Writing Efficient researcher Lega...
4      Computer Skills Proficient in MS office Word B...
5      Computer skills Yes SQL knowledge yes Unix kno...
                             ...                        
763    Skills ETL Data Warehousing SQL PL SQL Basic C...
800    Skill Sets Multi tasking Collaborative Optimis...
850    AREA OF EXPERTISE PROFILE Around 10 plus years...
870    Technical Skills Key Skills MS Technology Net ...
900    TECHNICAL SKILLS Programming Languages C NET W...
Name: Cleaned_text, Length: 184, dtype: object

In [ ]:
num_classes = df["Category"].nunique()
print(f"Number of Categories: {num_classes}\n")

Number of Categories: 25



In [ ]:
resumes=df['Cleaned_text'].values
sentences = ["[CLS] " + resume + " [SEP]" for resume in resumes]
labels = df['Category'].values

In [ ]:
# Generate tokens and input IDs for training
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased', do_lower_case=True)
tokenized_sentences = [tokenizer.tokenize(sentence) for sentence in sentences]

input_ids = [tokenizer.convert_tokens_to_ids(tokenized_sentence) for tokenized_sentence in tokenized_sentences ]
input_ids = pad_sequences(input_ids, maxlen=512, dtype="long", truncating="post", padding="post")
input_ids.shape

(184, 512)

In [ ]:
# Generate attention masks for training
attention_masks = [[float(i > 0) for i in input_id] for input_id in input_ids]

In [ ]:
# Split data into training and validation sets, use straified split due to class imbalance
training_inputs, validation_inputs, training_labels, validation_labels, training_masks,validation_masks = train_test_split(input_ids,labels, attention_masks, random_state=2018, test_size=0.15, stratify=labels)



# # Check the sizes of each set
print("Training Set Size:", len(training_inputs))
print("Validation Set Size:", len(validation_inputs))


Training Set Size: 156
Validation Set Size: 28


In [ ]:
batch_size=16
from sklearn.preprocessing import LabelEncoder

# Initialize LabelEncoder
label_encoder = LabelEncoder()

# Fit and transform the training labels
training_labels = label_encoder.fit_transform(training_labels)
validation_labels = label_encoder.transform(validation_labels)  # Use transform to maintain consistency

# Convert to numpy arrays of int64
training_labels = np.array(training_labels, dtype=np.int64)
validation_labels = np.array(validation_labels, dtype=np.int64)

# Ensure correct data types before converting to tensors
training_inputs = np.array(training_inputs, dtype=np.int64)  # Convert to int64
training_masks = np.array(training_masks, dtype=np.int64)      # Convert to int64
training_labels = np.array(training_labels, dtype=np.int64)    # Convert to int64

validation_inputs = np.array(validation_inputs, dtype=np.int64)  # Convert to int64
validation_masks = np.array(validation_masks, dtype=np.int64)      # Convert to int64
validation_labels = np.array(validation_labels, dtype=np.int64)    # Convert to int64

# Convert to PyTorch tensors
training_inputs = TensorDataset(
    torch.tensor(training_inputs, dtype=torch.long),
    torch.tensor(training_masks, dtype=torch.long),
    torch.tensor(training_labels, dtype=torch.long)
)

training_sampler = RandomSampler(training_inputs)
training_dataloader = DataLoader(training_inputs, sampler=training_sampler, batch_size=batch_size)

validation_inputs = TensorDataset(
    torch.tensor(validation_inputs, dtype=torch.long),
    torch.tensor(validation_masks, dtype=torch.long),
    torch.tensor(validation_labels, dtype=torch.long)
)

validation_sampler = SequentialSampler(validation_inputs)
validation_dataloader = DataLoader(validation_inputs, sampler=validation_sampler, batch_size=batch_size)

In [ ]:
dropout_rate = 0.1  # Set dropout rate

# Prepare model configuration
config = BertConfig.from_pretrained("bert-base-uncased", num_labels=25)
config.hidden_dropout_prob = dropout_rate  # Change dropout rate for BERT
config.attention_probs_dropout_prob = dropout_rate  # Dropout in attention layers (optional)

# Load BERT with modified dropout
model = BertForSequenceClassification.from_pretrained("bert-base-uncased", config=config)
model = nn.DataParallel(model)
model.to(device)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


DataParallel(
  (module): BertForSequenceClassification(
    (bert): BertModel(
      (embeddings): BertEmbeddings(
        (word_embeddings): Embedding(30522, 768, padding_idx=0)
        (position_embeddings): Embedding(512, 768)
        (token_type_embeddings): Embedding(2, 768)
        (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (encoder): BertEncoder(
        (layer): ModuleList(
          (0-11): 12 x BertLayer(
            (attention): BertAttention(
              (self): BertSdpaSelfAttention(
                (query): Linear(in_features=768, out_features=768, bias=True)
                (key): Linear(in_features=768, out_features=768, bias=True)
                (value): Linear(in_features=768, out_features=768, bias=True)
                (dropout): Dropout(p=0.1, inplace=False)
              )
              (output): BertSelfOutput(
                (dense): Linear(in_features=768, out_features=7

In [ ]:
# Prepare AdamW optimizer
parameters = list(model.named_parameters())
no_decay = ['bias', 'LayerNorm.weight']
parameters = [
{'params': [p for n, p in parameters if not any(nd in n for nd in no_decay)], 'weight_decay':
0.01},
{'params': [p for n, p in parameters if any(nd in n for nd in no_decay)], 'weight_decay': 0.0}
]
optimizer = AdamW(parameters, lr=0.00005, correct_bias=False)

/usr/local/lib/python3.10/dist-packages/transformers/optimization.py:640: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


In [ ]:
epochs = 50
# Compute class weights for imbalance
class_weights = compute_class_weight(class_weight="balanced", classes=np.unique(training_labels), y=training_labels)
class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)
loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights)

# Define optimizer and scheduler
# optimizer = AdamW(model.parameters(), lr=2e-5, betas=(0.9, 0.98), eps=1e-8, weight_decay=0.01)
num_training_steps = epochs * len(training_dataloader)
lr_scheduler = get_scheduler("linear", optimizer=optimizer, num_warmup_steps=100, num_training_steps=num_training_steps)

# Early Stopping Parameters
early_stopping_patience = 5
best_val_accuracy = 0
epochs_no_improve = 0

In [ ]:
# Calculate accuracy
def accuracy(predicted_labels, labels):
  predicted_labels = numpy.argmax(predicted_labels.to('cpu').numpy(), axis=1).flatten()
  labels = labels.to('cpu').numpy().flatten()
  return numpy.sum(predicted_labels == labels) / len(labels)

In [ ]:
# Training Loop with Gradient Accumulation
epochs = 50
training_losses = []
accumulation_steps = 2  # Simulates batch_size=32

for epoch in trange(epochs, desc="Epoch"):
    model.train()
    training_loss = 0
    training_steps = 0

    for step, batch in enumerate(training_dataloader):
        inputs = batch[0].to(device)
        attention_masks = batch[1].to(device)
        labels = batch[2].to(device)

        # Forward pass
        outputs = model(inputs, attention_mask=attention_masks, labels=labels)
        loss = loss_fn(outputs.logits, labels) / accumulation_steps  # Apply class weighting & scale loss
        loss.backward()

        # Perform optimizer step after accumulation_steps
        if (step + 1) % accumulation_steps == 0:
            optimizer.step()
            lr_scheduler.step()
            optimizer.zero_grad()

        training_loss += loss.item()
        training_steps += 1
        training_losses.append(loss.item())

    # Compute Average Training Loss
    average_training_loss = training_loss / training_steps
    print("Epoch {}: Average Training Loss: {}".format(epoch + 1, average_training_loss))

    # Validation Step
    model.eval()
    validation_accuracy = 0
    validation_steps = 0

    for batch in validation_dataloader:
        inputs = batch[0].to(device)
        attention_masks = batch[1].to(device)
        labels = batch[2].to(device)

        with torch.no_grad():
            outputs = model(inputs, attention_mask=attention_masks, labels=labels)
        logits = outputs.logits

        temp_validation_accuracy = accuracy(logits, labels)
        validation_accuracy += temp_validation_accuracy
        validation_steps += 1

    # Compute Average Validation Accuracy
    average_validation_accuracy = validation_accuracy / validation_steps
    print("Epoch {}: Validation Accuracy: {}".format(epoch + 1, average_validation_accuracy))

    # Early Stopping Check
    if average_validation_accuracy > best_val_accuracy:
        best_val_accuracy = average_validation_accuracy
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1

    if epochs_no_improve >= early_stopping_patience:
        print("Early stopping triggered! Stopping training.")
        break



Epoch:   0%|          | 0/50 [00:00<?, ?it/s]

Epoch 1: Average Training Loss: 1.63556569814682


Epoch:   2%|▏         | 1/50 [00:08<07:07,  8.73s/it]

Epoch 1: Validation Accuracy: 0.03125
Epoch 2: Average Training Loss: 1.6273747086524963


Epoch:   4%|▍         | 2/50 [00:17<06:58,  8.72s/it]

...

Epoch 23: Validation Accuracy: 0.9583333333333333
Epoch 24: Average Training Loss: 0.024233459495007992


Epoch:  48%|████▊     | 24/50 [03:30<03:48,  8.80s/it]

Epoch 24: Validation Accuracy: 0.9583333333333333
Epoch 25: Average Training Loss: 0.021984104439616202


Epoch:  48%|████▊     | 24/50 [03:39<03:57,  9.14s/it]

Epoch 25: Validation Accuracy: 0.9583333333333333
Early stopping triggered! Stopping training.


In [ ]:
resumes=df['Cleaned_text'].values
sentences = ["[CLS] " + resume + " [SEP]" for resume in resumes]
labels = df['Category'].values

In [ ]:
# Generate tokens and input IDs for evaluation
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased', do_lower_case=True)
tokenized_sentences = [tokenizer.tokenize(sentence) for sentence in sentences]
input_ids = [tokenizer.convert_tokens_to_ids(tokenized_sentence) for tokenized_sentence in tokenized_sentences]
input_ids = pad_sequences(input_ids, maxlen=256, dtype="long",truncating="post", padding="post")
input_ids = torch.tensor(input_ids)

In [ ]:
# Generate attention masks for evaluation
attention_masks = [[float(i > 0) for i in input_id] for input_id in input_ids]
attention_masks = torch.tensor(attention_masks)


In [ ]:
print(type(input_ids), type(attention_masks), type(labels))
print(input_ids.shape, attention_masks.shape, labels.shape)


<class 'torch.Tensor'> <class 'torch.Tensor'> <class 'numpy.ndarray'>
torch.Size([184, 256]) torch.Size([184, 256]) (184,)


In [ ]:
labels = label_encoder.fit_transform(labels)
labels = np.array(labels, dtype=np.int64)
labels = torch.tensor(labels, dtype=torch.long)

In [ ]:
# Prepare prediction dataset
prediction_dataset = TensorDataset(input_ids, attention_masks, labels)
prediction_dataloader = DataLoader(prediction_dataset, batch_size=batch_size)

# Evaluate model
model.eval()
logits_set = []
labels_set = []

for batch in prediction_dataloader:
    batch_input_ids, batch_attention_masks, batch_labels = batch  # Unpack correctly
    batch_input_ids = batch_input_ids.to(device)
    batch_attention_masks = batch_attention_masks.to(device)
    batch_labels = batch_labels.to(device)

    with torch.no_grad():
        outputs = model(batch_input_ids, attention_mask=batch_attention_masks)
        logits = outputs.logits

    logits_set.append(logits.cpu().numpy())
    labels_set.append(batch_labels.cpu().numpy())


In [ ]:
# Calculate Matthews correlation coefficient for each batch
matthews_set = []
for i in range(len(labels_set)):
  mcc = matthews_corrcoef(labels_set[i], numpy.argmax(logits_set[i], axis=1).flatten())
  matthews_set.append(mcc)
for i, mcc in enumerate(matthews_set):
  print(f"Batch {i + 1}: MCC = {mcc}")
# Calculate the overall Matthews correlation coefficient
overall_mcc = numpy.mean(matthews_set)
print(f"\nOverall MCC: {overall_mcc}")

Batch 1: MCC = 0.9353448275862069
Batch 2: MCC = 1.0
Batch 3: MCC = 1.0
Batch 4: MCC = 0.9356309347680319
Batch 5: MCC = 0.8035011108192193
Batch 6: MCC = 1.0
Batch 7: MCC = 0.9358974358974359
Batch 8: MCC = 1.0
Batch 9: MCC = 1.0
Batch 10: MCC = 1.0
Batch 11: MCC = 1.0
Batch 12: MCC = 1.0

Overall MCC: 0.9675311924225746
